# Import Libraries

In [1]:
import numpy as np
import pandas as pd 
from matplotlib import pyplot as plt

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv


# Read dataset

In [2]:
data = pd.read_csv('/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')
data.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


# EDA

In [3]:
data.info() # General structure check

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [4]:
data.isnull().sum() # Number of null values of each column

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [5]:
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

for col in categorical_cols:
    print(f"Values appear in the column '{col}':")
    print(data[col].value_counts())
    print("-" * 40)

Values appear in the column 'gender':
gender
Female    2994
Male      2115
Other        1
Name: count, dtype: int64
----------------------------------------
Values appear in the column 'ever_married':
ever_married
Yes    3353
No     1757
Name: count, dtype: int64
----------------------------------------
Values appear in the column 'work_type':
work_type
Private          2925
Self-employed     819
children          687
Govt_job          657
Never_worked       22
Name: count, dtype: int64
----------------------------------------
Values appear in the column 'Residence_type':
Residence_type
Urban    2596
Rural    2514
Name: count, dtype: int64
----------------------------------------
Values appear in the column 'smoking_status':
smoking_status
never smoked       1892
Unknown            1544
formerly smoked     885
smokes              789
Name: count, dtype: int64
----------------------------------------


# Preprocessing

In [6]:
# Fill median for null values
data['bmi'] = data['bmi'].fillna(data['bmi'].median())

# Delete observations that have values 'other' in Gender column
# Replace 'Other' with the most frequent value ('Female') directly
data['gender'] = data['gender'].replace('Other', 'Female')

# Convert values of categorical columns into 0/1
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
data_cleaned = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

In [7]:
data_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              5110 non-null   int64  
 1   age                             5110 non-null   float64
 2   hypertension                    5110 non-null   int64  
 3   heart_disease                   5110 non-null   int64  
 4   avg_glucose_level               5110 non-null   float64
 5   bmi                             5110 non-null   float64
 6   stroke                          5110 non-null   int64  
 7   gender_Male                     5110 non-null   bool   
 8   ever_married_Yes                5110 non-null   bool   
 9   work_type_Never_worked          5110 non-null   bool   
 10  work_type_Private               5110 non-null   bool   
 11  work_type_Self-employed         5110 non-null   bool   
 12  work_type_children              51

In [8]:
data_cleaned.isnull().sum()

id                                0
age                               0
hypertension                      0
heart_disease                     0
avg_glucose_level                 0
bmi                               0
stroke                            0
gender_Male                       0
ever_married_Yes                  0
work_type_Never_worked            0
work_type_Private                 0
work_type_Self-employed           0
work_type_children                0
Residence_type_Urban              0
smoking_status_formerly smoked    0
smoking_status_never smoked       0
smoking_status_smokes             0
dtype: int64

# Training 

## 1. Determine features and target

In [9]:
X = data_cleaned.drop(columns = ['stroke'])
Y = data_cleaned['stroke']

## 2. Split dataset

In [10]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

## 3. Train the model

In [11]:
regr = linear_model.LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42)
regr.fit(X_train, Y_train) # Find the best fit

LogisticRegression(class_weight='balanced', max_iter=10000, random_state=42)

## 4. Results

In [12]:
# Weights of the object function
print('Intercept: \n', regr.intercept_)
print('Coefficients: \n', regr.coef_)

Intercept: 
 [-5.55607893]
Coefficients: 
 [[ 5.04024964e-06  8.15647558e-02  5.11509697e-01  2.25254404e-01
   4.30184047e-03  7.46172600e-03 -1.15912620e-01 -2.15256726e-01
  -1.71984122e-01  1.68614735e-01 -1.20135456e-01  1.61542354e+00
  -5.11022784e-02 -9.76280266e-03 -1.90228397e-01  3.24231619e-01]]


# Test model

In [13]:
Y_pred = regr.predict(X_test)

# Convert continuous decimal predictions into binary labels (0 or 1) using a 0.5 threshold
Y_pred_binary = np.where(Y_pred >= 0.5, 1, 0)

# Generate the confusion matrix to see exact true/false counts
cm = confusion_matrix(Y_test, Y_pred_binary)

print("--- DETAILED MEDICAL EVALUATION ---")
print("Confusion Matrix:")
print(cm)
print("-" * 40)
print("Classification Report:")
print(classification_report(Y_test, Y_pred_binary, target_names=['No Stroke', 'Stroke'], zero_division=0))

--- DETAILED MEDICAL EVALUATION ---
Confusion Matrix:
[[711 249]
 [ 12  50]]
----------------------------------------
Classification Report:
              precision    recall  f1-score   support

   No Stroke       0.98      0.74      0.84       960
      Stroke       0.17      0.81      0.28        62

    accuracy                           0.74      1022
   macro avg       0.58      0.77      0.56      1022
weighted avg       0.93      0.74      0.81      1022

